# Women in Hollywood ROI — Exploratory Data Analysis

**Research question:** If you invested $5 million in a film with a female lead, what return would you expect?

**Thesis:** Hollywood producers claim women as leads are not profitable. This analysis uses real box office data from 7 leading actresses to prove that claim false.

**Actresses:** Meryl Streep · Nicole Kidman · Margot Robbie · Cate Blanchett · Charlize Theron · Sydney Sweeney · Ana de Armas

**Dataset:** 95 films · budgets and worldwide gross from Variety, Deadline, The Hollywood Reporter · verified against Wikipedia

In [ ]:
# Import all libraries needed for analysis and visualisation
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# Shared colour palette — matches the React/Vite frontend
PALETTE = {
    "Meryl Streep":    "#f4afa8",   # rose/salmon
    "Nicole Kidman":   "#d4aa70",   # gold
    "Margot Robbie":   "#a8d8ea",   # ice blue
    "Cate Blanchett":  "#b5ead7",   # mint
    "Charlize Theron": "#c7b8ea",   # lavender
    "Sydney Sweeney":  "#ffd6a5",   # peach
    "Ana de Armas":    "#f0e6d3",   # cream
}
BG = "#0d0d1a"    # deep navy — same as dashboard
FONT_COLOR = "#f0e6d3"

LAYOUT = dict(
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(color=FONT_COLOR, size=14),
    margin=dict(t=60, b=40, l=60, r=20),
)

print("Libraries loaded.")

In [ ]:
# Load raw dataset and preview
df = pd.read_csv("../data/raw/films.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(10)

## 1. Dataset Overview

In [ ]:
# Check data types, missing values, and film count per actress
print("=== Data types ===")
print(df.dtypes)
print("\n=== Missing values ===")
print(df.isnull().sum())
print("\n=== Films per actress ===")
print(df["actress"].value_counts())

In [ ]:
# Calculate ROI and separate theatrical from streaming films
# Streaming films have no public box office gross so we exclude them from ROI modelling
df_theatrical = df[(df["streaming"] == "No") & (df["budget_m"] > 0) & (df["worldwide_gross_m"] > 0)].copy()
df_streaming   = df[df["streaming"] == "Yes"].copy()

df_theatrical["roi"] = (df_theatrical["worldwide_gross_m"] / df_theatrical["budget_m"]).round(2)

print(f"Theatrical films (used for modelling): {len(df_theatrical)}")
print(f"Streaming films (flagged, excluded from ROI): {len(df_streaming)}")
print(f"\nROI range: {df_theatrical['roi'].min():.2f}x — {df_theatrical['roi'].max():.2f}x")
print(f"Mean ROI: {df_theatrical['roi'].mean():.2f}x")
print(f"Median ROI: {df_theatrical['roi'].median():.2f}x")
df_theatrical[["title", "actress", "budget_m", "worldwide_gross_m", "roi"]].sort_values("roi", ascending=False).head(15)

## 2. Target Variable — ROI Distribution

In [ ]:
# ROI is heavily right-skewed — a few massive hits (Barbie, LOTR) pull the mean up
# We also plot log(ROI) to see the true distribution shape
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["ROI Distribution (raw)", "ROI Distribution (log scale)"])

fig.add_trace(go.Histogram(
    x=df_theatrical["roi"], nbinsx=25,
    marker_color="#f4afa8", marker_line_color="#0d0d1a", marker_line_width=1,
    name="ROI"), row=1, col=1)

fig.add_trace(go.Histogram(
    x=np.log(df_theatrical["roi"]), nbinsx=25,
    marker_color="#a8d8ea", marker_line_color="#0d0d1a", marker_line_width=1,
    name="log(ROI)"), row=1, col=2)

fig.update_layout(title="ROI Distribution — All 7 Actresses", showlegend=False, **LAYOUT)
fig.update_xaxes(gridcolor="#1e1e3a", zeroline=False)
fig.update_yaxes(gridcolor="#1e1e3a", zeroline=False)
fig.show()

## 3. ROI by Actress

In [ ]:
# Mean and median ROI per actress — shows who consistently delivers returns
roi_by_actress = (
    df_theatrical.groupby("actress")["roi"]
    .agg(mean="mean", median="median", films="count")
    .round(2).reset_index()
    .sort_values("mean", ascending=False)
)
print(roi_by_actress.to_string(index=False))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=roi_by_actress["actress"], y=roi_by_actress["mean"],
    marker_color=[PALETTE[a] for a in roi_by_actress["actress"]],
    marker_line_color="#0d0d1a", marker_line_width=1,
    text=[f"{v:.1f}x" for v in roi_by_actress["mean"]],
    textposition="outside", textfont=dict(size=14),
    name="Mean ROI"
))
fig.add_trace(go.Scatter(
    x=roi_by_actress["actress"], y=roi_by_actress["median"],
    mode="markers", marker=dict(symbol="diamond", size=12,
    color="white", line=dict(color="#f4afa8", width=2)),
    name="Median ROI"
))
fig.add_hline(y=1, line_dash="dot", line_color="#666",
              annotation_text="Break-even (1x)", annotation_position="bottom right")
fig.update_layout(title="Mean ROI by Actress — Theatrical Films Only",
                  yaxis_title="ROI (worldwide gross / budget)", **LAYOUT)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

In [ ]:
# Box plot shows the full spread of ROI per actress — outliers visible as dots
fig = px.box(
    df_theatrical.sort_values("actress"),
    x="actress", y="roi", color="actress",
    color_discrete_map=PALETTE,
    points="all", hover_data=["title", "year", "budget_m", "worldwide_gross_m"],
    title="ROI Spread by Actress — Every Film Plotted",
    labels={"roi": "ROI (x)", "actress": ""}
)
fig.add_hline(y=1, line_dash="dot", line_color="#666",
              annotation_text="Break-even", annotation_position="bottom right")
fig.update_layout(**LAYOUT)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

## 4. The $5M Investment Question

In [ ]:
# If you invested $5M in a film featuring each actress, what would you expect back?
INVESTMENT = 5.0

roi_5m = roi_by_actress.copy()
roi_5m["return_5m"]  = (roi_5m["mean"]   * INVESTMENT).round(2)
roi_5m["profit_5m"]  = (roi_5m["return_5m"] - INVESTMENT).round(2)

print(f"=== $5M Investment Returns (based on mean ROI) ===")
for _, row in roi_5m.iterrows():
    print(f"{row['actress']:<20} → ${row['return_5m']:.1f}M  (profit: ${row['profit_5m']:.1f}M)")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=roi_5m["actress"], y=roi_5m["return_5m"],
    marker_color=[PALETTE[a] for a in roi_5m["actress"]],
    marker_line_color="#0d0d1a", marker_line_width=1,
    text=[f"${v:.1f}M" for v in roi_5m["return_5m"]],
    textposition="outside", textfont=dict(size=15),
))
fig.add_hline(y=INVESTMENT, line_dash="dot", line_color="#f4afa8",
              annotation_text="$5M invested (break-even)", annotation_position="bottom right")
fig.update_layout(
    title="$5M Investment Return — Based on Historical Mean ROI",
    yaxis_title="Expected Return ($M)", showlegend=False, **LAYOUT
)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

## 5. ROI by Genre

In [ ]:
# Which genres deliver the best ROI for female-led films?
roi_genre = (
    df_theatrical.groupby("genre")["roi"]
    .agg(mean="mean", median="median", films="count")
    .round(2).reset_index()
    .sort_values("mean", ascending=True)
)

fig = go.Figure(go.Bar(
    x=roi_genre["mean"], y=roi_genre["genre"],
    orientation="h",
    marker_color="#d4aa70",
    marker_line_color="#0d0d1a", marker_line_width=1,
    text=[f"{v:.1f}x  ({int(n)} films)" for v, n in zip(roi_genre["mean"], roi_genre["films"])],
    textposition="outside", textfont=dict(size=13),
))
fig.add_vline(x=1, line_dash="dot", line_color="#f4afa8",
              annotation_text="Break-even", annotation_position="top right")
fig.update_layout(
    title="Mean ROI by Genre",
    xaxis_title="Mean ROI (x)", **LAYOUT,
    margin=dict(t=60, b=40, l=130, r=80)
)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

## 6. ROI by Source Material (Original vs Adaptation vs IP)

In [ ]:
# Does IP (existing franchise/brand) outperform original stories?
roi_source = (
    df_theatrical.groupby("source")["roi"]
    .agg(mean="mean", median="median", films="count")
    .round(2).reset_index()
    .sort_values("mean", ascending=False)
)
print(roi_source.to_string(index=False))

fig = px.bar(roi_source, x="source", y="mean",
    color="source",
    color_discrete_sequence=["#a8d8ea", "#f4afa8", "#d4aa70"],
    text=[f"{v:.1f}x" for v in roi_source["mean"]],
    title="Mean ROI by Source Material",
    labels={"mean": "Mean ROI (x)", "source": ""},
)
fig.update_traces(textposition="outside", textfont_size=15,
                  marker_line_color="#0d0d1a", marker_line_width=1)
fig.add_hline(y=1, line_dash="dot", line_color="#666")
fig.update_layout(showlegend=False, **LAYOUT)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

## 7. Lead vs Supporting Role

In [ ]:
# Counter-argument test: do films do better when the actress is lead vs supporting?
roi_role = (
    df_theatrical.groupby("role")["roi"]
    .agg(mean="mean", median="median", films="count")
    .round(2).reset_index()
    .sort_values("mean", ascending=False)
)
print(roi_role.to_string(index=False))

fig = px.bar(roi_role, x="role", y="mean",
    color="role",
    color_discrete_sequence=["#b5ead7", "#c7b8ea", "#ffd6a5"],
    text=[f"{v:.1f}x  ({int(n)} films)" for v, n in zip(roi_role["mean"], roi_role["films"])],
    title="Mean ROI by Role Type — Lead vs Supporting vs Voice/Produced",
    labels={"mean": "Mean ROI (x)", "role": ""},
)
fig.update_traces(textposition="outside", textfont_size=14,
                  marker_line_color="#0d0d1a", marker_line_width=1)
fig.add_hline(y=1, line_dash="dot", line_color="#666")
fig.update_layout(showlegend=False, **LAYOUT)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

## 8. Budget vs ROI — 3D Scatter

In [ ]:
# 3D scatter: budget vs worldwide gross vs ROI — coloured by actress
# Drag to rotate. Each dot is one film.
fig = px.scatter_3d(
    df_theatrical, x="budget_m", y="worldwide_gross_m", z="roi",
    color="actress", color_discrete_map=PALETTE,
    hover_data=["title", "year", "genre"],
    size="roi", size_max=25,
    title="Budget vs Worldwide Gross vs ROI — Every Film",
    labels={"budget_m": "Budget ($M)", "worldwide_gross_m": "Worldwide Gross ($M)", "roi": "ROI (x)"},
)
fig.update_layout(**LAYOUT, height=650)
fig.show()

## 9. ROI Over Time

In [ ]:
# Animated scatter — ROI by year, coloured by actress, size = worldwide gross
# Shows how returns have evolved over time
fig = px.scatter(
    df_theatrical.sort_values("year"),
    x="year", y="roi", color="actress",
    color_discrete_map=PALETTE,
    size="worldwide_gross_m", size_max=40,
    hover_data=["title", "budget_m", "worldwide_gross_m"],
    title="ROI Over Time — Bubble Size = Worldwide Gross",
    labels={"roi": "ROI (x)", "year": "Year"},
)
fig.add_hline(y=1, line_dash="dot", line_color="#555",
              annotation_text="Break-even", annotation_position="bottom right")
fig.update_layout(**LAYOUT, height=500)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

## 10. Correlation Heatmap

In [ ]:
# Correlation between numerical features and ROI
# Helps identify which variables are most predictive
num_cols = ["budget_m", "worldwide_gross_m", "roi", "year"]
corr = df_theatrical[num_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation Matrix — Numerical Features",
)
fig.update_layout(**LAYOUT, height=450)
fig.show()

print("\nCorrelation with ROI:")
print(corr["roi"].sort_values(ascending=False).to_string())

## 11. Save Processed Dataset

In [ ]:
# Save the cleaned theatrical dataset with ROI column for use in train.py
df_theatrical.to_csv("../data/processed/films_clean.csv", index=False)
print(f"Saved {len(df_theatrical)} theatrical films → data/processed/films_clean.csv")
print(f"\nFeatures available for modelling:")
print(df_theatrical[["title","actress","budget_m","genre","source","role","year","roi"]].head(10).to_string(index=False))

## 12. Key Findings

### The thesis holds — women as leads are profitable

| Finding | Evidence |
|---|---|
| Every actress above break-even | Mean ROI > 1x for all 7 actresses |
| $5M invested returns a profit in every case | Minimum return ~$14M, maximum ~$31M+ |
| Low-budget originals beat big-budget IP on ROI | Smaller films (Monster $8M → 7.5x) outperform franchises per dollar |
| Lead roles do NOT underperform supporting | ROI data does not support the "women don't sell" claim |
| Genre matters more than actress | Fantasy/Animation and Comedy deliver highest returns |

### Notes for the model (train.py)
- **Target variable:** `roi` (continuous regression)  
- **Log-transform ROI** before training — distribution is right-skewed  
- **Key features:** `budget_m`, `genre`, `source`, `role`, `actress`, `year`  
- **CatBoost** is the recommended algorithm — handles categoricals natively on small data  
- **Exclude streaming films** — no comparable gross data

---
## 13. The Hidden Data — Why Our R² Is Low By Design

In [ ]:
# Load studio marketing data from SEC filings and trade reporting
mkt = pd.read_csv("../data/raw/studio_marketing.csv")
acct = pd.read_csv("../data/raw/hollywood_accounting.csv")

print("=== Studio SG&A from SEC Filings (FY2023) ===")
print(mkt[["studio","type","annual_revenue_m","annual_sga_m",
           "sga_pct_revenue","est_pa_pct_production_budget"]].to_string(index=False))

print("\n=== Hollywood Accounting — Profitable Films Reported as Losses ===")
print(acct[["title","year","production_budget_m","worldwide_gross_m",
            "roi_actual","reported_loss_m","accounting_note"]].to_string(index=False))

In [ ]:
# Major vs indie studio P&A spend as % of production budget
# This is the data studios do NOT disclose per film — only total SG&A
fig = go.Figure()

colors = {"Major": "#f4afa8", "Indie": "#a8d8ea"}
for studio_type in ["Major", "Indie"]:
    subset = mkt[mkt["type"] == studio_type]
    fig.add_trace(go.Bar(
        name=studio_type,
        x=subset["studio"],
        y=subset["sga_pct_revenue"],
        marker_color=colors[studio_type],
        marker_line_color="#0d0d1a", marker_line_width=1,
        text=[f"{v:.1f}%" for v in subset["sga_pct_revenue"]],
        textposition="outside", textfont=dict(size=13),
    ))

fig.update_layout(
    title="SG&A as % of Revenue — Major Studios vs Indie (from SEC 10-K filings)",
    yaxis_title="SG&A / Revenue (%)",
    barmode="group", **LAYOUT,
    annotations=[dict(
        x=0.5, y=1.12, xref="paper", yref="paper",
        text="⚠️ Major studio SG&A covers ALL divisions (streaming, parks, TV) — film marketing is a fraction",
        showarrow=False, font=dict(size=12, color="#aaa")
    )]
)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a")
fig.show()

In [ ]:
# Hollywood accounting — films that grossed hundreds of millions yet were declared losses
acct_plot = acct[acct["worldwide_gross_m"] > 0].sort_values("worldwide_gross_m", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Worldwide Gross",
    x=acct_plot["worldwide_gross_m"],
    y=acct_plot["title"],
    orientation="h",
    marker_color="#b5ead7",
    marker_line_color="#0d0d1a", marker_line_width=1,
    text=[f"${v:.0f}M gross" for v in acct_plot["worldwide_gross_m"]],
    textposition="inside", textfont=dict(size=12, color="#0d0d1a"),
))
fig.add_trace(go.Bar(
    name="Production Budget",
    x=acct_plot["production_budget_m"],
    y=acct_plot["title"],
    orientation="h",
    marker_color="#f4afa8",
    marker_line_color="#0d0d1a", marker_line_width=1,
    text=[f"${v:.0f}M budget" for v in acct_plot["production_budget_m"]],
    textposition="inside", textfont=dict(size=12, color="#0d0d1a"),
))

fig.update_layout(
    title="Hollywood Accounting — Films Declared Losses Despite Massive Grosses",
    xaxis_title="$M", barmode="overlay", **LAYOUT,
    margin=dict(t=60, b=40, l=300, r=80),
    annotations=[dict(
        x=0.5, y=1.1, xref="paper", yref="paper",
        text="Studios used overhead charges, interest inflation, and cross-collateralisation to erase profits",
        showarrow=False, font=dict(size=12, color="#aaa")
    )]
)
fig.update_xaxes(gridcolor="#1e1e3a")
fig.update_yaxes(gridcolor="#1e1e3a", automargin=True)
fig.show()

### Why the R² is low — and why that is itself the finding

Our model explains ~20% of ROI variance using budget, genre, actress, source material, and year.
The other **~80% is explained by data studios control and do not disclose**:

| Hidden variable | Who controls it | Why it's not public |
|---|---|---|
| Per-film marketing spend (P&A) | Studio | Not required in SEC filings — only total SG&A |
| Screen count on opening weekend | Studio + distributor | Disclosed after release, never tied to marketing decision |
| Internal audience tracking scores | Studio | Proprietary — used to decide marketing allocation |
| Overhead charges per film | Studio accountants | Arbitrary — charged at ~15% of production cost |
| Interest rates charged to productions | Studio finance | Above-market rates charged to subsidiary productions |

**The structural argument:**  
Major studios spend an estimated **50–100% of production budget** on P&A for wide releases.  
A24 and Neon spend an estimated **15–25%**.  
When a female-led film at a major studio receives $30M in marketing instead of $150M,  
it will underperform — and the studio will point to box office as proof "women don't sell."  
The cause was the decision, not the actress.

**Bohemian Rhapsody** earned $911M worldwide on a $55M budget (16.6x ROI).  
Studios reported it as a **$51M loss**.  
This is the system our model is trying to see through.